<a href="https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/Test_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install Ollama binary into the Colab Linux system
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (3,865 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122412 files and directories currently

In [ ]:
# 2. Boot the Ollama server engine in the background using a Python thread
import threading
import subprocess
import time

def launch_ollama_server():
    # Launches 'ollama serve' without blocking the notebook cell execution
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Spin up the thread
server_thread = threading.Thread(target=launch_ollama_server, daemon=True)
server_thread.start()

# Give the engine a few seconds to warm up and bind to port 11434
time.sleep(5)
print("🚀 Ollama Server is successfully running in the Colab background!")

🚀 Ollama Server is successfully running in the Colab background!


In [ ]:
!/usr/local/bin/ollama --version

ollama version is 0.24.0


In [ ]:
# 3. Pull the specific model you want to use (e.g., Qwen 2.5 7B or Llama 3.2 3B)
!/usr/local/bin/ollama pull qwen2.5:7b

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)

GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

In [ ]:
import ollama
import time

def ask_colab_ollama(prompt_text):
    print("Processing request via Colab's T4 GPU...\n")

    start_time = time.time()
    full_response = ""
    # We use streaming so you see the response come alive chunk by chunk
    stream = ollama.generate(
        model='qwen2.5:7b',
        prompt=prompt_text,
        stream=True
    )

    for chunk in stream:
        response_chunk = chunk['response']
        full_response += response_chunk
        print(response_chunk, end='', flush=True)

    end_time = time.time()
    elapsed_time = end_time - start_time

    print(f"\n\nOllama LLM Response Time: {elapsed_time:.2f} seconds")
    return full_response, elapsed_time

### Test Model Summarization

Let's ask the `qwen2.5:7b` model to summarize the content of the local `README.md` file found in `/content/sample_data/`.

In [ ]:
with open('/content/sample_data/README.md', 'r') as f:
    readme_content = f.read()

prompt = f"""Please summarize the following text:

{readme_content}
"""

ask_colab_ollama(prompt)

Processing request via Colab's T4 GPU...

This directory contains several sample datasets to help you get started:

- `california_housing_data*.csv`: This dataset includes California housing data from the 1990 US Census. For more details, visit: <https://docs.google.com/document/d/e/2PACX-1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia_DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN/pub>

- `mnist_*.csv`: This is a small sample from the MNIST database, which is described at: <http://yann.lecun.com/exdb/mnist/>

- `anscombe.json`: Contains a copy of Anscombe's quartet. Originally described in Anscombe, F. J. (1973), and prepared by the vega_datasets library (<https://github.com/altair-viz/vega_datasets/blob/4f67bdaad10f45e3549984e17e1b3088c731503d/vega_datasets/_data/anscombe.json>).

### Clone a Public GitHub Repository

You can clone a public GitHub repository into your Colab environment using the `!git clone` command. Replace `YOUR_PUBLIC_REPOSITORY_URL` with the URL of the repository you wish to clone.

In [ ]:
!git clone YOUR_PUBLIC_REPOSITORY_URL

fatal: repository 'YOUR_PUBLIC_REPOSITORY_URL' does not exist


### Accessing Private GitHub Repositories

To clone a private GitHub repository, you'll need to authenticate. Here are two common methods:

1.  **Using SSH Keys:**
    *   Generate an SSH key pair in Colab (if you don't have one). `!ssh-keygen -t rsa -b 4096 -C "your_email@example.com"`
    *   Add the public key (`~/.ssh/id_rsa.pub`) to your GitHub account settings.
    *   Configure SSH for GitHub in Colab. `!ssh-keyscan github.com >> ~/.ssh/known_hosts`
    *   Then clone using the SSH URL: `!git clone git@github.com:username/repo.git`

2.  **Using a GitHub Personal Access Token (PAT):**
    *   Generate a PAT on GitHub with `repo` scope permissions. Store it securely (e.g., Colab secrets).
    *   You can then clone using the token in the URL: `!git clone https://<YOUR_PAT>@github.com/username/repo.git`
    *   Alternatively, you can configure git to use the PAT. For example, `!git config --global credential.helper store` then `!git clone https://github.com/username/repo.git` (it will prompt for username/password, use PAT as password).

In [ ]:
# Example using Personal Access Token (PAT) from Colab Secrets
# 1. Go to the '🔑' icon on the left panel (Secrets tab).
# 2. Add a new secret with the name 'GITHUB_TOKEN' and paste your PAT as the value.
# 3. Enable 'Notebook access' for this secret.

from google.colab import userdata
import os

# Get the PAT from Colab secrets
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

# Set up git to use the PAT (this is temporary for the session)
!git config --global url."https://{GITHUB_TOKEN}@github.com/".insteadOf "https://github.com/"

# Now you can clone your private repository using its standard HTTPS URL
# Replace 'your_username' and 'your_private_repo' with your actual GitHub username and repository name.
!git clone https://github.com/Chanakya1998begin/trapX.git

# You can also unset the credential helper if you wish, for security
# !git config --global --unset url."https://{GITHUB_TOKEN}@github.com/".insteadOf

Cloning into 'trapX'...
remote: Enumerating objects: 11268, done.
remote: Counting objects: 100% (266/266), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 11268 (delta 141), reused 145 (delta 114), pack-reused 11002 (from 2)
Receiving objects: 100% (11268/11268), 48.07 MiB | 20.88 MiB/s, done.
Resolving deltas: 100% (3664/3664), done.


Remember to replace `your_username` and `your_private_repo.git` with your actual GitHub details. For SSH, replace `your_email@example.com` and use your SSH clone URL.

### Summarize content from `trapX/project/llm_advisory/skills/counter_fibo_verdict.md`

In [ ]:
file_path = '/content/trapX/project/llm_advisory/skills/counter_fibo_verdict.md'

try:
    with open(file_path, 'r') as f:
        file_content = f.read()

    summary_prompt = f"""Please summarize the following document:

{file_content}
"""

    ask_colab_ollama(summary_prompt)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the path is correct and the repository was cloned successfully.")
except Exception as e:
    print(f"An error occurred: {e}")

Processing request via Colab's T4 GPU...

Sure, please provide the specific context or data related to the counter you want me to analyze (e.g., price movements, jerks, institutional trades, etc.). This will help in generating the exact output format you need.

In [ ]:
!/usr/local/bin/ollama list

NAME          ID              SIZE      MODIFIED      
qwen2.5:7b    845dbda0ea48    4.7 GB    3 minutes ago    


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -R /content/drive/MyDrive/

/content/drive/MyDrive/:
 2024
 8500288124_EqOpenPositions.gsheet
 8500288124_EqOpenPositions.xls
 8500288124_PortFolioEqtAll.gsheet
 8500288124_PortFolioEqtAll.xls
'Agentic AI Toxicology Report SaaS.gdoc'
'AI Impact on Generic Strategy'$'\n''.gdoc'
'Analyse all the data in tables in the attached si....gdoc'
'An Expert Review of the Strategic Intelligence Report on the Rimegepant (Nurtec ODT) Generic Opportunity.gdoc'
'Any any missing section from the attached two doc... (1).gsheet'
'Any any missing section from the attached two doc... (2).gsheet'
'Any any missing section from the attached two doc... (3).gsheet'
'Any any missing section from the attached two doc....gsheet'
 Apr-Trades_24Apr_to_21May_2023_PnL.gsheet
'A Spiral-Based Strategic Guide to Momentum, Force, and Impulse.gdoc'
'A Strategic Playbook for Generic Drug Market Entry: A Case Study on Rimegepant (Nurtec ODT).gdoc'
'AZD5462 Retrosynthesis and Commercial Viability.gdoc'
'Batch Process Optimization for AZD5462.gdoc'
'Bemp

In [ ]:
log_file_path = '/content/drive/MyDrive/VM-4-logs/May_14/trapx_20260514_090139.log'

try:
    with open(log_file_path, 'r') as f:
        log_content = f.read()

    # The content of counter_fibo_verdict.md is already in the 'file_content' variable
    # We will use it as a system-level instruction for the model.
    combined_prompt = f"""{file_content}

Here is the log data for the 11:20 minute bar processing:

{log_content}

Based on the log data provided, please provide your verdict for the 11:20 minute bar processing as a senior institutional-trading advisor, following the 'Counter-Fibo 100% Verdict Advisory' guidelines. Focus specifically on the data relevant to the 11:20 minute bar."""

    print("Sending combined prompt to Ollama model...")
    ask_colab_ollama(combined_prompt)
except FileNotFoundError:
    print(f"Error: The log file '{log_file_path}' was not found. Please ensure the path is correct.")
except Exception as e:
    if "[Errno 111] Connection refused" in str(e):
        print(f"An error occurred: The Ollama server appears to be disconnected or not running. Please try re-running the cell that launches the Ollama server (cell ID vvAfIx1xaeuI) and then re-execute this cell.")
    else:
        print(f"An unexpected error occurred while processing the log file: {e}")

Sending combined prompt to Ollama model...
Processing request via Colab's T4 GPU...

An error occurred: The Ollama server appears to be disconnected or not running. Please try re-running the cell that launches the Ollama server (cell ID vvAfIx1xaeuI) and then re-execute this cell.


In [ ]:
# Restarting the Ollama server to restore connection
import threading
import subprocess
import time

def launch_ollama_server():
    # Launches 'ollama serve' without blocking the notebook cell execution
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Spin up the thread
server_thread = threading.Thread(target=launch_ollama_server, daemon=True)
server_thread.start()

# Give the engine a few seconds to warm up and bind to port 11434
time.sleep(5)
print("🚀 Ollama Server is successfully running in the Colab background!")

🚀 Ollama Server is successfully running in the Colab background!


In [ ]:
log_file_path = '/content/drive/MyDrive/VM-4-logs/May_14/trapx_20260514_090139.log'

try:
    with open(log_file_path, 'r') as f:
        log_content = f.read()

    # The content of counter_fibo_verdict.md is already in the 'file_content' variable
    # We will use it as a system-level instruction for the model.
    combined_prompt = f"""{file_content}

Here is the log data for the 11:20 minute bar processing:

{log_content}

Based on the log data provided, please provide your verdict for the 11:20 minute bar processing as a senior institutional-trading advisor, following the 'Counter-Fibo 100% Verdict Advisory' guidelines. Focus specifically on the data relevant to the 11:20 minute bar. Format your output exactly as follows:

━━━━━━━━━━━━━━━━━━━━━━
🤖 LLM advisory:
Verdict: [Your Verdict Score, a value between -1.00 (extreme bearish) and +1.00 (extreme bullish)]
Action:
* [Action Point 1]
* [Action Point 2]
* [Action Point 3]
Dtls: [Detailed explanation]"""

    print("Sending combined prompt to Ollama model...")
    _, response_time = ask_colab_ollama(combined_prompt)
except FileNotFoundError:
    print(f"Error: The log file '{log_file_path}' was not found. Please ensure the path is correct.")
except Exception as e:
    if "[Errno 111] Connection refused" in str(e):
        print(f"An error occurred: The Ollama server appears to be disconnected or not running. Please try re-running the cell that launches the Ollama server (cell ID vvAfIx1xaeuI) and then re-execute this cell.")
    else:
        print(f"An unexpected error occurred while processing the log file: {e}")

Sending combined prompt to Ollama model...
Processing request via Colab's T4 GPU...

━━━━━━━━━━━━━━━━━━━━━━
🤖 LLM advisory:
Verdict: -0.50 (moderately bearish)
Action:
* Re-evaluate the trade setup given the failure of key delta ladder conditions.
* Consider selling pressure indicated by TWAP below 23600 levels.
* Monitor for further downside movement, especially in lower delta options.

Dtls: 
The advisory is moderately bearish based on the following observations from the log data:
1. The TWAP at 15:29 was around 23630.93, indicating overall market sentiment favoring a downward trend.
2. The Delta ladder analysis shows significant conflict in both up and down sides, with no clear bullish setup identified.
3. Key trade criteria were not met as indicated by the failure to start the Visualizer server on port 9001, suggesting system issues or lack of strong market signals.
4. Overall, the log data suggests a bearish bias but does not confirm an extreme bearish stance, hence a score of -0.